# Phase 5 — a group's past, and the horizon it goes stale across

Every per-group feature the harness could build averaged the group's rows
together: the mean target (`fit_encoding`), the mean/median/spread of some other
column (`fit_group_stats`). All of them throw away the **order** the rows came
in. Nothing in the vocabulary could say *what this store sold most recently*,
which is a statement about **when**, not about **how much**.

This notebook builds that, by hand first. The hard part is not storing the past —
it is deciding what the past means for a row the model has never seen, sitting
some unknown distance after everything you know. Two decision records answer that:

| record | question | answer |
|---|---|---|
| **0047** | what does a lag serve past the end of stored history? | the last known value **plus its age**, and age is per row |
| **0048** | may a validation row read its own neighbours' labels? | no — history is frozen per fold at the training prefix |

Section 6 is the one that matters. Everything before it is machinery.

## 1. The data: a panel where recency actually matters

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)


def make_panel(n_days=120):
    """A store-day panel whose sales *wander* rather than sit at a level.

    Each store's sales follow a random walk. That matters for this notebook:
    if sales were a fixed per-store level plus noise, `fit_group_stats` would
    already capture everything and there would be nothing for history to add.
    Here, *when* you last saw a store tells you more than what it usually sells.
    """
    level = {"s1": 30.0, "s2": 34.0, "s3": 38.0, "s4": 42.0}
    start = pd.Timestamp("2015-01-01")
    rows = []
    for store, base in level.items():
        walk = base + np.cumsum(rng.normal(scale=1.5, size=n_days))
        for i, d in enumerate(pd.date_range(start, periods=n_days)):
            rows.append({"store": store, "date": d,
                         "promo": int(rng.random() > 0.7),
                         "sales": walk[i]})
    return pd.DataFrame(rows).sort_values(["date", "store"]).reset_index(drop=True)


panel = make_panel()
panel.head()

## 2. Why this needed a tool

The obvious way to write "yesterday's value" is one line of pandas. Try it in the
free-code path.

In [ ]:
from tabular_agent.transform import transform_features

CODE = """
def transform_features(df):
    df = df.copy()
    df["promo_yesterday"] = df.groupby("store")["promo"].shift(1)
    return df
"""

try:
    transform_features(panel, CODE)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:300]}")

That is 0011's stability check, and it is **right**. Only the *code* is stored, so
at predict time it re-runs `shift(1)` over whatever rows arrived in that request.
One row alone gets `NaN`; the same row batched with its neighbours gets something
else. The feature depends on who it was batched with, which is not a property a
feature may have.

The same refusal fires for the target, on top of the target being invisible to
transforms at all (0009):

In [ ]:
CODE_TARGET = """
def transform_features(df):
    df = df.copy()
    df["sales_yesterday"] = df.groupby("store")["sales"].shift(1)
    return df
"""

try:
    transform_features(panel, CODE_TARGET)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:300]}")

So the capability was blocked from both directions, and correctly from both. What
makes it legal is not different arithmetic — it is that the number comes from
**stored state** rather than from the frame in front of it. That is the same thing
that makes `fit_group_stats` legal where `groupby().transform("mean")` is not.

## 3. `HistoryEncoder` by hand

Fit it on the whole panel and look at what it kept.

In [ ]:
from tabular_agent.encoders import HistoryEncoder

enc = HistoryEncoder(["store"], "sales", "date").fit(panel)

print("history ends at:", pd.Timestamp("1970-01-01") + pd.Timedelta(days=enc.frozen_at_))
print("columns:        ", enc.output_columns)
print("stored per group:", {k: len(v[0]) for k, v in enc.history_.items()})

One observation per store, because nothing asked for more. Now serve it rows from
after the end of history — the real case, where the model is asked about days it
has never seen.

In [ ]:
future = pd.DataFrame({
    "store": ["s1"] * 4,
    "date": pd.to_datetime(["2015-05-01", "2015-05-05", "2015-05-15", "2015-07-01"]),
})

# Indexed by date so the frame being rounded is all numbers.
enc.transform(future).set_index("date").round(2)

**The value never changes. The age does.** That pair is the whole design (0047).

Serving the value alone would be a carry-forward: the same number, drifting
further from the truth the further ahead you predict, and saying nothing about
it. This project's characteristic failure is a number that looks fine and
describes something else, and a bare carry-forward is that failure in feature
form. The age column is what lets a model discount a value it should not trust.

And because the answer comes from storage, it does not depend on the batch:

In [ ]:
alone = enc.transform(future.iloc[[3]])
batch = enc.transform(future)

print("alone:", round(alone["store_sales_latest"].iloc[0], 2), "age", alone["store_sales_latest_ago"].iloc[0])
print("batch:", round(batch["store_sales_latest"].iloc[3], 2), "age", batch["store_sales_latest_ago"].iloc[3])

## 4. Storage is bounded, on purpose

An encoder lives in a table's lineage and is carried in every bundle. If its size
grew with the length of the table, it could not. `keep` is a per-group cap, so the
encoder is (number of groups × `keep`) and never a function of how many rows
there are.

In [ ]:
big = make_panel(n_days=2000)

for keep in (1, 8):
    e = HistoryEncoder(["store"], "sales", "date", keep=keep).fit(big)
    stored = sum(len(v[0]) for v in e.history_.values())
    print(f"keep={keep}: {stored:>3} observations stored, for {len(big)} rows of input")

## 5. Real lags, and the horizon that makes them legal

`_latest` needs no declaration to be honest. A *lag* does. "The value 7 days ago" is
only answerable from stored history if every row you are asked about is at least 7
days past the end of it — so the task has to say how far ahead it forecasts.

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.encoders import fit_history

folds = make_folds(panel["sales"], "regression", n_splits=4,
                   order=panel["date"].to_numpy())

try:
    fit_history(panel, ["store"], "sales", "date", folds=folds, offsets=[1])
except ValueError as e:
    print("REFUSED:", e)

`TaskSpec.forecast_horizon` is where that declaration goes — a fact about the
serving contract, which makes it the data owner's to state (0029). It is a
**minimum**, not a constant: within one predict batch the rows sit at different
distances from the end of history, so it cannot be a single number applied to
every row. Per-row distance is what `_latest_ago` reports. The horizon's only job is
to say which offsets are **legal**.

**Legal is not the same as useful**, and that distinction cost Rossmann v8 a
table (0050). An offset shorter than the *whole period you predict* is answerable
for the earliest rows and, for the rest, quietly falls back to the last-known
value. The declared floor is the gap to the *first* row; what an offset actually
needs to clear is the *span of the block*. `fit_history` reports
`offsets_answerable` for exactly this reason.


In [ ]:
try:
    fit_history(panel, ["store"], "sales", "date", folds=folds,
                offsets=[1], forecast_horizon=7)
except ValueError as e:
    print("REFUSED:", e)

At or beyond the declared horizon, an offset is legal. And when it cannot be
answered exactly — a row far past the end of history — it degrades into `_latest`
and **reports the age it actually found**, rather than a stale number under a name
that promises a fresh one:

In [ ]:
enc7 = HistoryEncoder(["store"], "sales", "date", offsets=[7], keep=10).fit(panel)

out = enc7.transform(future)
out.set_index("date")[["store_sales_7_ago", "store_sales_7_ago_exact"]].round(2)

The first two rows get a genuine 7-day lag, so `_7_ago_exact` is 1. The last two
are far enough out that "7 days ago" is past the end of history, so they get the
newest thing available and `_exact` drops to **0** — the column is saying it has
become `_latest` under a name that promises a fixed lag.

A flag rather than an age here, because for an offset the age is near-constant —
it equals the offset whenever the lookup worked — so all of its information is in
the failures. `_latest` keeps a real age, which genuinely varies row to row.

## 6. The per-fold freeze — the part that matters (0048)

Everything so far was about *predict* time. Now: what may a **validation** row
read?

`docs/time_aware_features.md` §3.4 framed this as a leakage question and reached
for out-of-fold construction, the guard `fit_group_stats` uses. That framing is
wrong, and 0048 exists to say why. Two different failures wear the same word:

1. **Self-inclusion** — the row's own target inside its own feature. This is what
   out-of-fold removes. A strictly backward lag **cannot commit it**: the row at
   `t` only ever reads `t-k`. Out-of-fold buys nothing here.
2. **Availability skew** — reading a label that will not exist when it is served.
   Real, severe, and *not* what out-of-fold fixes.

So the guard a target lag needs is the **horizon, not the fold**. The mechanism
still ends up fold-shaped — each fold's history is frozen at the end of its
training prefix — but for the second reason, which changes what a correct
implementation looks like.

In [ ]:
oof, final = fit_history(panel, ["store"], "sales", "date", folds=folds)

frozen = oof["store_sales_latest"]
frozen_age = oof["store_sales_latest_ago"]

view = pd.concat([panel[["store", "date", "sales"]], oof], axis=1)
view[view["store"] == "s1"].iloc[22:32].set_index("date").round(2)

Rows in the first block keep `NaN` — under forward chaining nothing precedes them
to train on, the same gap out-of-fold target encoding leaves.

Then look at what happens *inside* a block: the value stands still and the age
climbs. The feature goes stale across the validation block exactly as it would go
stale across a real forecast horizon.

In [ ]:
_, va = folds[-1]
s1_rows = va[(panel.iloc[va]["store"] == "s1").to_numpy()]

print("age across the last validation block:")
print(frozen_age.iloc[s1_rows].dropna().astype(int).tolist())

Compare that with the naive backward lag — the one that reads whatever row
happens to sit before it, regardless of fold.

In [ ]:
naive = panel.groupby("store")["sales"].shift(1)
seen = frozen.notna() & naive.notna()

# Compared *within* store: the between-store level differences would otherwise
# make any store-flavoured number look near-perfect and hide the whole effect.
dev = panel["sales"] - panel.groupby("store")["sales"].transform("mean")


def within_store_corr(feature):
    f = feature - feature.groupby(panel["store"]).transform("mean")
    return round(float(np.corrcoef(f[seen], dev[seen])[0, 1]), 3)


print(f"naive lag-1 : age always 1, corr with y = {within_store_corr(naive)}")
print(f"frozen      : age up to {int(frozen_age.max()):>2}, corr with y = {within_store_corr(frozen)}")

**The honest feature looks much worse, and it is supposed to.**

The naive lag validates beautifully and cannot be served: at serving time nobody
hands you yesterday's label for a day you are being asked to forecast. The gap
between those two numbers is **the size of the lie, not the size of the loss**.

0048 states this explicitly so it is not later mistaken for a bug and "fixed".

## 7. Why random folds are refused outright

Under forward chaining, "frozen at the end of the training fold" is a prefix in
time and age counts real elapsed days. Under random k-fold the training fold is
scattered *through* time, so a validation row's most recent stored observation is
whichever neighbour happened to land in the other fold.

In [ ]:
random_folds = make_folds(panel["sales"], "regression", n_splits=4)

try:
    fit_history(panel, ["store"], "sales", "date", folds=random_folds)
except ValueError as e:
    print("REFUSED:", e)

The number is computable. Here is what it would have been — the ages a validation
row would see under random folds, against the ages forward chaining produces:

In [ ]:
tr, va_rand = random_folds[0]
ages = []
for r in va_rand:
    same = [i for i in tr if panel["store"][i] == panel["store"][r]
            and panel["date"][i] < panel["date"][r]]
    if same:
        newest = max(same, key=lambda i: panel["date"][i])
        ages.append((panel["date"][r] - panel["date"][newest]).days)

print("random folds    :", pd.Series(ages).value_counts().sort_index().to_dict())
print("forward chaining:", frozen_age.iloc[s1_rows].dropna().astype(int).tolist())

That contrast is the argument. Random folds teach the model that **yesterday is
almost always available** — four rows in five sit at age 1. Forward chaining, and
real serving, look like `1, 2, 3, … 24`: age grows smoothly across the horizon.

Change the seed and both the value and its age change, because the holes punched
in history are an artifact of the RNG. That is a *different distribution*, not a
noisier one — which is why re-running with another seed does not converge on the
right answer, and why this is a refusal (0013) rather than a warning. A warning
that does not block is a warning that gets ignored.

## 8. Through the harness

Same machinery, driven by the tool the agent actually calls. The ordering column
is the one the run is split on; there has to be one, which is the same refusal
seen from the other side.

In [ ]:
import os
import tempfile

from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import finalize_run, fit_history as fit_history_tool, train_tabular_model
from tabular_agent.pipeline import load_bundle, predict_bundle
from tabular_agent.taskspec import TaskSpec

run_root = tempfile.mkdtemp()
src = os.path.join(run_root, "panel.csv")
panel.to_csv(src, index=False)

session = RunSession(
    os.path.join(run_root, "run"), src,
    TaskSpec(target="sales", task_type="regression", metric="rmse", n_splits=4),
    split_on="date",
)

out = fit_history_tool(session, by=["store"], value="sales")
print({k: out[k] for k in ("table", "new_columns", "n_groups", "n_rows_without_value")})
print()
print(out["note"])

Note the tool may point at **the target**, where `fit_group_stats` refuses it. A
per-group *statistic* of the target is target encoding and has its own tool; a
per-group *most recent* target is neither, and it cannot be smuggled out of a
served frame either — the target is not in one, so it can only ever come from
stored history.

Now train, finalize, and ask the bundle about days after everything it ever saw:

In [ ]:
train_tabular_model(session, table=out["table"])
fin = finalize_run(session)
bundle = load_bundle(os.path.join(session.run_dir, fin["bundle_dir"]))

future_rows = pd.DataFrame({
    "store": ["s1", "s2", "s1"],
    "date": pd.to_datetime(["2015-05-02", "2015-05-02", "2015-06-15"]),
    "promo": [0, 1, 0],
})

future_rows.assign(prediction=predict_bundle(bundle, future_rows).round(2))

The lineage replayed the stored history and answered rows dated weeks past the
last row it was fitted on — with no `sales` column in the request, because there
never is one.

## What is left

- **Slice 3: rolling and expanding windows.** Once history is stored and replayed,
  windows are aggregations over the same tail. `keep` already carries the `L + W`
  bound they need.
- **Known future covariates.** A promo calendar genuinely *is* knowable at `T+5`,
  and freezing it throws away real information. 0048 rejects the shortcut of
  inferring this from "is it the target" and leaves the declaration unbuilt.
- **Recursive forecasting** — feeding predictions back as lags — stays rejected
  (0047), reopenable with evidence.
- **None of this has been validated by a run.** The estimate of what it is worth
  is one number on one dataset.